# HRDiT → Modular Diffusers — parity + publish (v2, self-contained block)

`HRDiTLadderBlock` is now **one self-contained block** (encodes the prompt itself; all 7 components sourced
from FLUX.1-dev) — exactly the template shape, so it publishes as `block.py` + `modular_config.json`
(no weights, no `modular_model_index`) and loads via `auto_map` + `trust_remote_code`.

**v1 bug (fixed):** the old `save_pretrained` push wrote component weights but **no `block.py`**, so
`from_pretrained(trust_remote_code=True)` fell back to stock FLUX → single 2048² denoise → OOM.

**Run order:** A (parity, prove ladder==monolith) → C (publish PRIVATE, code-only) → D (load from Hub).
**Runtime:** A100-80GB · `HUGGINGFACE_TOKEN` (remyxai) · accept FLUX.1-dev license · **upload `block.py`**.

## 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf torchvision

## 2 · GPU + HF auth (remyxai)

In [ ]:
import torch
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
from huggingface_hub import login, whoami
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
print("orgs:", [o["name"] for o in whoami().get("orgs", [])])

## 3 · Load the self-contained `block.py` (upload it first)

In [ ]:
import os, importlib
assert os.path.exists("block.py"), "Upload the NEW self-contained block.py, then re-run."
import block as B; importlib.reload(B)
print("loaded:", [n for n in dir(B) if n.startswith("HRDiT")])

## 4 · Milestone A — PARITY (gate): ladder == our pipeline (prompt-based, reduced config)
Both encode the same prompt internally (identical embeds) + same seed. Monolith first, freed, then modular.

In [ ]:
import urllib.request, gc, torch
from diffusers import FluxPipeline
REPO, DEV, DT, SEED = "black-forest-labs/FLUX.1-dev", "cuda", torch.bfloat16, 0
PROMPT = "an alpine meadow at golden hour"
CFG = dict(height=1024, width=1024, resolutions=[512, 1024], ntk_factor=[4.0], spa_steps=[0],
           alphas=[0.0], betas=[0.0], num_inference_steps=8, num_inference_steps_highres=[6],
           guidance_scale=3.5, guidance_scale_highres=[4.5], output_type="latent")

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/smellslikeml/diffusers/flux-hrdit/examples/community/pipeline_flux_hrdit.py",
    "hrdit_pipeline.py")
import hrdit_pipeline as Mono
stock = FluxPipeline.from_pretrained(REPO, torch_dtype=DT).to(DEV)
hp = Mono.HRDiTFluxPipeline(**stock.components)
g = torch.Generator(DEV).manual_seed(SEED)
l_mono = hp(PROMPT, generator=g, **CFG).images.float().cpu()
del hp, stock; gc.collect(); torch.cuda.empty_cache()

pipe = B.HRDiTLadderBlock().init_pipeline()      # components resolve from ComponentSpec (FLUX.1-dev)
pipe.load_components(dtype=DT); pipe.to(DEV)
g = torch.Generator(DEV).manual_seed(SEED)
out = pipe(prompt=PROMPT, generator=g, **CFG)
l_mod = (out.images if hasattr(out, "images") else out).float().cpu()
d = (l_mono - l_mod).abs().max().item()
print(f"[MILESTONE A] ladder-vs-monolith max|Δ| = {d:.3e}  ->  {'PASS' if d < 5e-2 else 'FAIL'}")

## 5 · Milestone C — publish PRIVATE, code-only (delete the broken v1 repo first)

In [ ]:
import json
from huggingface_hub import HfApi
REPO_ID = "remyxai/hrdit-flux-modular"
FLUX = "black-forest-labs/FLUX.1-dev"

open("modular_config.json", "w").write(json.dumps({
    "_class_name": "HRDiTLadderBlock", "_diffusers_version": "0.41.0.dev0",
    "auto_map": {"ModularPipelineBlocks": "block.HRDiTLadderBlock"}}, indent=2))

def comp(sub, lib, cls):
    return [None, None, {"pretrained_model_name_or_path": FLUX, "revision": None,
                         "subfolder": sub, "type_hint": [lib, cls], "variant": None}]
open("modular_model_index.json", "w").write(json.dumps({
    "_blocks_class_name": "HRDiTLadderBlock", "_class_name": "ModularPipeline",
    "_diffusers_version": "0.41.0.dev0",
    "text_encoder":   comp("text_encoder",   "transformers", "CLIPTextModel"),
    "tokenizer":      comp("tokenizer",      "transformers", "CLIPTokenizer"),
    "text_encoder_2": comp("text_encoder_2", "transformers", "T5EncoderModel"),
    "tokenizer_2":    comp("tokenizer_2",    "transformers", "T5TokenizerFast"),
    "transformer":    comp("transformer",    "diffusers",    "FluxTransformer2DModel"),
    "vae":            comp("vae",            "diffusers",    "AutoencoderKL"),
    "scheduler":      comp("scheduler",      "diffusers",    "FlowMatchEulerDiscreteScheduler")}, indent=2))

api = HfApi()
api.create_repo(REPO_ID, private=True, repo_type="model", exist_ok=True)
# IMPORTANT: do NOT upload README.md here — the rich model card (hero + gallery) is managed
# separately; re-uploading a stub here would clobber it. Publish only code + config.
for f in ("block.py", "modular_config.json", "modular_model_index.json"):
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO_ID)
print("published code/config:", api.list_repo_files(REPO_ID))

## 6 · Milestone D — load from the Hub via trust_remote_code + run

In [ ]:
import gc, torch
try: del pipe
except Exception: pass
gc.collect(); torch.cuda.empty_cache()
from diffusers import ModularPipeline
hub = ModularPipeline.from_pretrained("remyxai/hrdit-flux-modular", trust_remote_code=True)
print("loaded block:", type(hub.blocks).__name__)     # expect HRDiTLadderBlock, NOT FluxAutoBlocks
hub.load_components(dtype=DT); hub.to(DEV)
g = torch.Generator(DEV).manual_seed(SEED)
img = hub(prompt="a photograph of a mountain lake at dawn", height=1024, width=1024, output_type="pil").images[0]
img.save("hrdit_from_hub.png"); print("loaded-from-hub run OK -> hrdit_from_hub.png (try height=2048 next)")

## 7 · Milestone E — 2048² full-ladder parity sweep (NTK + SPA + structure)
The last gate: compare the **published Hub pipeline** vs the monolith at 2048² with DEFAULT schedules
(ntk `[4,10]`, spa `[3,0]`, structure on) — this exercises the upscale stage that 1024² skipped.
Memory-safe: monolith first, freed, then the Hub pipeline. `PASS` here clears the repo to go public.

In [ ]:
import gc, os, urllib.request, torch
from diffusers import FluxPipeline, ModularPipeline
DEV, DT, SEED = "cuda", torch.bfloat16, 0
PROMPT = "an alpine meadow at golden hour"
# 2048 -> ladder [1024, 2048]: base 1024 (stock) + one NTK/SPA/structure upscale stage.
# reduced steps keep it fast; ntk/spa/alphas/betas left as DEFAULTS so all three features are ON.
SWEEP = dict(height=2048, width=2048, num_inference_steps=8, num_inference_steps_highres=[6], output_type="latent")

for v in ("pipe", "hub", "stock", "hp", "full", "l_mono", "l_hub"):
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()

if not os.path.exists("hrdit_pipeline.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/smellslikeml/diffusers/flux-hrdit/examples/community/pipeline_flux_hrdit.py",
        "hrdit_pipeline.py")
import hrdit_pipeline as Mono

# monolith reference
stock = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=DT).to(DEV)
hp = Mono.HRDiTFluxPipeline(**stock.components)
g = torch.Generator(DEV).manual_seed(SEED)
l_mono = hp(PROMPT, generator=g, **SWEEP).images.float().cpu()
del hp, stock; gc.collect(); torch.cuda.empty_cache()

# published community modular pipeline, from the Hub
hub = ModularPipeline.from_pretrained("remyxai/hrdit-flux-modular", trust_remote_code=True)
hub.load_components(dtype=DT); hub.to(DEV)
g = torch.Generator(DEV).manual_seed(SEED)
out = hub(prompt=PROMPT, generator=g, **SWEEP)
l_hub = (out.images if hasattr(out, "images") else out).float().cpu()

d = (l_mono - l_hub).abs().max().item()
print(f"[MILESTONE E] 2048 full-ladder (NTK+SPA+structure) hub-vs-monolith max|Δ| = {d:.3e}  ->  {'PASS' if d < 5e-2 else 'FAIL'}")
# for a visual, re-run the hub line with output_type='pil' and .images[0].save(...)

## 7 · What green licenses
- **A PASS** = ladder faithful. **C** = `block.py` bundled (check `list_repo_files` shows block.py + modular_config.json).
- **D**: `loaded block: HRDiTLadderBlock` (not FluxAutoBlocks) + a real image = the published community modular pipeline works end-to-end.
- Then: parity sweep (structure ON `alphas=[1.0],betas=[0.5]`, SPA ON `spa_steps=[3]`, full 4096), flip repo public, post the maintainer ack.

## 8 · 4K showcase (optional) — full HRDiT ladder from the published pipeline
`height=width=4096` → ladder `1024→2048→4096` (two NTK/SPA/structure upscale stages). ~1–2 min, ~26 GB.

In [ ]:
import gc, torch
from diffusers import ModularPipeline
for v in ("stock", "hp", "l_mono", "l_hub"):
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()
if "hub" not in globals():
    hub = ModularPipeline.from_pretrained("remyxai/hrdit-flux-modular", trust_remote_code=True)
    hub.load_components(dtype=torch.bfloat16); hub.to("cuda")
g = torch.Generator("cuda").manual_seed(0)
img = hub(prompt="an alpine meadow at golden hour, snow-capped peaks, ultra detailed",
          height=4096, width=4096, output_type="pil").images[0]   # defaults: ntk [4,10], spa [3,0]
img.save("hrdit_4k.png"); print("saved", img.size, "-> hrdit_4k.png")

## 9 · Peak VRAM at default 4K (FLUX.1-dev) — the exact number for the HRDiT ref issue
Measures true peak (weights + activations) at 4096² on defaults, no HAP.

In [ ]:
import torch, gc
from diffusers import ModularPipeline
gc.collect(); torch.cuda.empty_cache()
if "hub" not in globals():
    hub = ModularPipeline.from_pretrained("remyxai/hrdit-flux-modular", trust_remote_code=True)
    hub.load_components(dtype=torch.bfloat16); hub.to("cuda")
hub.vae.enable_tiling(); hub.vae.enable_slicing()          # REQUIRED for 4K: VAE decode/encode dominate otherwise
base = torch.cuda.memory_allocated() / 1e9
torch.cuda.reset_peak_memory_stats()
g = torch.Generator("cuda").manual_seed(0)
_ = hub(prompt="an alpine meadow at golden hour, snow-capped peaks", height=4096, width=4096)  # pil -> includes the 4K decode
print("[PEAK VRAM @ 4096^2 . FLUX.1-dev . bf16 . VAE tiling ON]")
print(f"  weights resident : {base:.1f} GB")
print(f"  peak reserved    : {torch.cuda.max_memory_reserved()/1e9:.1f} GB   (must fit on the card)")
print("  NOTE: without vae.enable_tiling(), the 4096^2 VAE decode+encode alone pushes peak toward the full 80 GB.")